# 🚨🚨🚨  READ THIS FIRST  🚨🚨🚨
-----------------------------------------

### This cell runs the **full denoising pipeline** end-to-end.

**Estimated runtime: >1 hour on Google Colab T100 GPU.**  
Use it when you have time to let the full training complete (e.g. overnight).

---

## Full Training Pipeline (Long Runtime)

This section runs the **complete Noise2Noise denoising pipeline** end-to-end:

- load XRF detector-frame data,  
- explore and curate the image stack,  
- assemble the training dataset,  
- configure the U-Net and training schedule,  
- train for **up to 1400 epochs**,  
- compute loss/PSNR curves,  
- run inference on all saved checkpoints.

---

## Copyright & License

**© 2025 Dmitry Karpov**  
Assistant Professor of Physics and Materials Science, Université Grenoble Alpes  
(Materials Modelling and Exploration Laboratory, MEM – UGA/CEA)

These notebooks are part of the **DIADEM Academy** training series on U-Net and image analysis  
and are provided as **educational material**.

Distributed under the **Creative Commons Attribution–NonCommercial 4.0 International License (CC BY-NC 4.0)**  
https://creativecommons.org/licenses/by-nc/4.0/

You may **use, adapt, and modify** this material for **non-commercial research and teaching**,  
with proper attribution to the author.

Commercial use is **not permitted** without prior written permission.

Please cite when using or adapting this material:

**D. Karpov,  
*U-Net for Image Analysis — DIADEM Academy Training Materials*,  
Université Grenoble Alpes, 2025–present.**

*Provided “as is”, without warranty of any kind.*

---

In [ ]:
# this line is just to check GPU status
!nvidia-smi

In [ ]:
# this cell is needed to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import Button, IntSlider, HBox, VBox, HTML, Layout, Output
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable

import tensorflow as tf
from tensorflow.keras import mixed_precision


# mixed precision for speed on Colab GPUs
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

In [ ]:
# set the path to the setkafluo library in the directory that
# you cloned with the notebooks
# Be careful! Sometimes Colab needs change of My Drive to MyDrive or vice versa
sys.path.append("/content/drive/MyDrive/setkafluo_demo/notebooks_and_library/libs/")

import setkafluo as skf
print("SetkaFluo version:", skf.__version__)

from setkafluo.denoise import (
    extract_covering_patches_with_overlap_pad,
    generate_noise2noise_samples,
    standardize_images,
    extract_random_patches,
    augment_patch,
)


In [ ]:
# The path to the base folder
base_path = Path("/content/drive/MyDrive/setkafluo_demo/")

training_path = base_path / "training/" # your path where the training will happen
data_dir  = base_path / "input_data/" # path to the data

# We will define ckpt_dir / pred_dir right after
# selecting `element` in the next cell
def make_output_dirs(element: str):
    month = datetime.now().strftime("%b")
    ckpt_dir = training_path / f"training/{element}/{month}"
    pred_dir = training_path / f"training/{element}/predictions"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    pred_dir.mkdir(parents=True, exist_ok=True)
    return ckpt_dir, pred_dir

In [ ]:
# Pick the element you want to work with (e.g. "K-K", "Zn-K", "Fe-K", …)
element = "Ir-L"

# Create output directories for this element
ckpt_dir, pred_dir = make_output_dirs(element)

def read_tif_files(directory_path, ardesia=True, element='Zn-K', elements_to_discard=None):
    """
    Returns:
        idxs (np.ndarray of int)
        images (list[np.ndarray])  <-- keep as list to avoid object-dtype arrays
        names (np.ndarray of str)
    """
    suffix = "_ardesia" if ardesia else "_arwen"
    records = []  # (idx, image, filename)

    for fname in os.listdir(directory_path):
        if fname.endswith(f"_{element}_area_density_ngmm2.tif") and "_element" in fname and suffix in fname:
            i0 = fname.index("_element") + len("_element")
            i1 = fname.index("_", i0)
            idx = int(fname[i0:i1])
            if elements_to_discard and idx in elements_to_discard:
                continue
            img = skf.load_tiff(os.path.join(directory_path, fname))
            records.append((idx, img, fname))

    # sort by index
    records.sort(key=lambda t: t[0])

    if not records:
        return np.array([], dtype=int), [], np.array([], dtype=str)

    idxs  = np.array([r[0] for r in records], dtype=int)
    imgs  = [r[1] for r in records]                 # list of np.ndarrays
    names = np.array([r[2] for r in records], dtype=str)
    return idxs, imgs, names


In [ ]:
# Interactive visualization: click/slide through the stack with indices and filenames shown
data_dir_stack = data_dir / "processed/separate_elements/"

# Load the full stack (no exclusions yet)
idxs, images_stack, names = read_tif_files(str(data_dir_stack), ardesia=True, element=element)
print(f"Loaded {len(images_stack)} frames for {element}. Indexes: {idxs}")

if len(images_stack) == 0:
    print("No images found for the selected element.")
else:
    def to_numeric_img(x):
        arr = np.asarray(x)
        if arr.dtype == object:
            arr = arr.astype(np.float32)
        return arr

    def auto_contrast(img):
        p1, p99 = np.percentile(img, [1, 99])
        if p1 == p99:
            p1, p99 = float(np.min(img)), float(np.max(img))
        return p1, p99

    # Widgets
    slider = IntSlider(value=0, min=0, max=len(images_stack)-1, step=1,
                       description='Frame:', continuous_update=False)
    btn_prev = Button(description="⟵ Prev", layout=Layout(width="100px"))
    btn_next = Button(description="Next ⟶", layout=Layout(width="100px"))
    info = HTML()
    controls = HBox([btn_prev, slider, btn_next])
    ui = VBox([controls, info])

    out = Output()

    def draw(k: int):
        img = to_numeric_img(images_stack[k])
        vmin, vmax = auto_contrast(img)

        out.clear_output(wait=True)
        with out:
            # Larger figure so the image is bigger
            fig, ax = plt.subplots(figsize=(8.5, 8.5))
            im = ax.imshow(img, vmin=vmin, vmax=vmax)  # default colormap
            ax.set_title(f"{element}  •  detector index = {idxs[k]}", fontsize=12)
            ax.set_axis_off()

            # Colorbar with same height as the image axis
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="3%", pad=0.05)
            cb = fig.colorbar(im, cax=cax)

            # Ensure colorbar height matches the image height exactly
            ax.set_box_aspect(img.shape[0] / img.shape[1])

            fig.tight_layout()
            plt.show()

        info.value = (
            f"<code>{k+1}/{len(images_stack)}</code> | "
            f"idx=<b>{idxs[k]}</b> | "
            f"min={np.min(img):.3g}, mean={np.mean(img):.3g}, max={np.max(img):.3g}"
        )

    def on_prev(_):
        if slider.value > slider.min:
            slider.value -= 1
            draw(slider.value)

    def on_next(_):
        if slider.value < slider.max:
            slider.value += 1
            draw(slider.value)

    def on_slide(change):
        if change["name"] == "value":
            draw(change["new"])

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    slider.observe(on_slide)

    display(ui, out)
    draw(slider.value)

In [ ]:
# After browsing above, set the indexes you want to exclude
bad_elements = [15]

# Reload the stack without the bad indexes
idxs, images_stack, names = read_tif_files(
    str(data_dir_stack), ardesia=True, element=element, elements_to_discard=bad_elements
)

print("Indexes:", idxs)
print("Files:", names)

# Ensure we have a proper NumPy stack for training: (N, H, W)
if isinstance(images_stack, list):
    try:
        images_stack = np.stack(images_stack, axis=0)  # will fail if shapes differ
    except ValueError as e:
        # If this happens, your images are different sizes.
        # You can either resize/pad them or drop the offenders.
        raise ValueError(
            f"Could not stack images into a single array because shapes differ. "
            f"Please inspect and fix the inconsistent images. Original error: {e}"
        )

print("Stack shape:", images_stack.shape, images_stack.dtype)

In [ ]:
# Evaluation (weighted sum) image
data_dir_weight = data_dir / "processed/weighted_sums/"

fluo1 = skf.load_tiff(
    data_dir_weight / f"IMG_fluo1_{element}_area_density_ngmm2.tif"
)
print("Eval image:", fluo1.shape, fluo1.dtype)

# Visualize with matched colorbar height
p1, p99 = np.percentile(fluo1, [1, 99])
if p1 == p99:
    p1, p99 = float(np.min(fluo1)), float(np.max(fluo1))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
im = ax.imshow(fluo1, vmin=p1, vmax=p99)  # default colormap
ax.set_title(f"Weighted sum {element}")
ax.set_axis_off()

# Make colorbar axis with same height as image axis
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.05)
cb = fig.colorbar(im, cax=cax)

# Preserve image aspect so the colorbar height matches
ax.set_box_aspect(fluo1.shape[0] / fluo1.shape[1])

fig.tight_layout()
plt.show()

In [ ]:
# Hyperparameters
PATCH  = 32
BATCH  = 4
EPOCHS = 1400
STEPS  = 128

cfg = skf.DenoiseConfig(
    lr=1e-5,
    batch_size=BATCH,
    patch_size=PATCH,
    steps_per_epoch=STEPS,
    epochs=EPOCHS,
    # model
    input_channels=1,
    output_channels=1,
    start_ch=64,
    depth=4,
    inc_rate=2.0,
    dropout=0.0,
    instancenorm=False,
    averagepool=False,
    upconv=False,
    residual=False,
    lambda_reg=0.0,
    reg_l1=False,
    # inference
    min_overlap=PATCH // 2,               # half-patch overlap
    # checkpoints
    save_models_dir=str(ckpt_dir),
    save_every_epochs=50,
    # runtime
    mixed_precision_warn=True,
)

cfg

In [ ]:
model, history = skf.train(images_stack, cfg)

In [ ]:
hist = history.history
epochs = range(1, len(hist['loss']) + 1)

# Loss
plt.figure(figsize=(6,4))
plt.plot(epochs, hist['loss'], label='loss')
if 'val_loss' in hist: plt.plot(epochs, hist['val_loss'], label='val_loss')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.grid(True)
plt.show()

# PSNR (key name can vary)
psnr_key = next((k for k in hist.keys() if k.lower().endswith('psnr') and not k.lower().startswith('val_')), None)
if psnr_key:
    plt.figure(figsize=(6,4))
    plt.plot(epochs, hist[psnr_key], label=psnr_key)
    vpsnr_key = f"val_{psnr_key}" if f"val_{psnr_key}" in hist else None
    if vpsnr_key: plt.plot(epochs, hist[vpsnr_key], label=vpsnr_key)
    plt.xlabel('epoch'); plt.ylabel('PSNR (dB)'); plt.legend(); plt.grid(True)
    plt.show()

In [ ]:
print("Evaluating on fluo1 with all checkpoints…")
skf.predict_all_checkpoints(
    checkpoints_dir=str(ckpt_dir),
    out_dir=str(pred_dir),
    img2d=fluo1,
    cfg=cfg,
    delete_after=False,  # keep checkpoints
)

In [ ]:
denoised = skf.predict_tiled(fluo1, model, cfg)
skf.save_tiff(denoised, pred_dir / "prediction_final_model.tif")
print("Saved:", pred_dir / "prediction_final_model.tif")